# Analyze Unmatched Facts from WikiGap Data

This notebook reads annotation files and extracts facts that are not present in both language versions.

In [2]:
# Step 1: Load all annotation files from the directory

import json
from pathlib import Path
import os
from openai import OpenAI
from dotenv import load_dotenv
import time
from datetime import datetime

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("THE_KEY"))

# Define paths
data_dir = Path("scratch/annotation_save/wikigap_data")
output_dir = Path("scratch/annotation_save/analysis_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Get all annotation files
annotation_files = sorted(list(data_dir.glob("annotation_*.json")))

print(f"Found {len(annotation_files)} annotation files")
print(f"Output directory: {output_dir}")
print(f"\nFirst 5 files:")
for i, file in enumerate(annotation_files[:5], 1):
    print(f"  {i}. {file.name}")

Found 362 annotation files
Output directory: scratch/annotation_save/analysis_outputs

First 5 files:
  1. annotation_2025-11-07_Armand Sabatier_fr.json
  2. annotation_2025-11-07_Augustin Jacob Landré-Beauvais_fr.json
  3. annotation_2025-11-07_Auriana Lazraq-Khlass_fr.json
  4. annotation_2025-11-07_Clarisse Coignet_fr.json
  5. annotation_2025-11-07_Cécile Charrier_fr.json


In [3]:
# Step 2: Process annotations and extract unmatched facts

# Language to country/culture mapping
LANG_COUNTRY_MAPPING = {
    'en': 'English-speaking countries (US, UK, etc.)',
    'fr': 'France',
    'it': 'Italy',
    'de': 'Germany',
    'es': 'Spain',
    'ru': 'Russia',
    'zh': 'China',
    'ja': 'Japan',
    'ar': 'Arab countries',
    'pt': 'Portugal/Brazil',
    'nl': 'Netherlands',
    'pl': 'Poland',
    'ko': 'Korea',
}

def extract_article_name(filename):
    """Extract article name from filename like 'annotation_2025-11-07_Armand Sabatier_fr.json'"""
    # Remove 'annotation_' prefix and date
    parts = filename.replace('annotation_', '').split('_')
    # Remove date (first part after annotation_)
    parts = parts[1:]
    # Remove language suffix (_fr, _en, etc)
    name_parts = '_'.join(parts).rsplit('_', 1)[0]
    return name_parts.replace('.json', '')

def extract_unmatched_facts_from_file(file_path):
    """Extract unmatched facts from a single annotation file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    columns = {col['name']: col['values'] for col in data['columns']}
    
    facts = columns['fact']
    languages = columns['language']
    intersection_labels = columns['intersection_label']
    person_name = columns['person_name'][0]
    
    # Group unmatched facts by language
    unmatched = {}
    for fact, lang, label in zip(facts, languages, intersection_labels):
        if label == 'no':
            if lang not in unmatched:
                unmatched[lang] = []
            # Remove duplicates
            if fact not in unmatched[lang]:
                unmatched[lang].append(fact)
    
    return person_name, unmatched

# Process all files and collect unmatched facts
all_unmatched = {}

print("Processing annotation files...\n")
for i, file_path in enumerate(annotation_files, 1):
    article_name = extract_article_name(file_path.name)
    person_name, unmatched = extract_unmatched_facts_from_file(file_path)
    
    if unmatched:  # Only add if there are unmatched facts
        all_unmatched[article_name] = unmatched
        total_facts = sum(len(facts) for facts in unmatched.values())
        print(f"{i}. {article_name}: {total_facts} unmatched facts across {len(unmatched)} languages")

print(f"\n=== SUMMARY ===")
print(f"Total articles with unmatched facts: {len(all_unmatched)}")
print(f"Languages found: {set(lang for article in all_unmatched.values() for lang in article.keys())}")

Processing annotation files...

1. Armand Sabatier: 28 unmatched facts across 2 languages
2. Augustin Jacob Landré-Beauvais: 54 unmatched facts across 2 languages
3. Auriana Lazraq-Khlass: 14 unmatched facts across 2 languages
4. Clarisse Coignet: 23 unmatched facts across 2 languages
5. Cécile Charrier: 11 unmatched facts across 2 languages
6. Faronne Ollivier: 2 unmatched facts across 1 languages
7. Hohenzollern Redoubt action, 2–18 March 1916: 106 unmatched facts across 2 languages
8. Jacques François Perroud: 25 unmatched facts across 2 languages
9. Joseph Brummer: 1 unmatched facts across 1 languages
10. Lancaster's chevauchée of 1346: 91 unmatched facts across 2 languages
11. Louis-Charles Couturier: 5 unmatched facts across 2 languages
12. Marie-Léontine Bordes-Pène: 2 unmatched facts across 2 languages
13. Maurice de Talleyrand-Périgord: 34 unmatched facts across 2 languages
14. Michel Soutif: 81 unmatched facts across 2 languages
16. Renée Vivien Prize: 16 unmatched facts acro

In [6]:
all_unmatched

{'Armand Sabatier': {'fr': ["Il fait ses études à Montpellier, où il suit les cours de mathématiques spéciales au lycée, puis s'inscrit en médecine.",
   'Il donne ces cours à la faculté de théologie protestante de Montauban.',
   'Il épouse Laure Gervais de Rouville, ils ont une fille, Jeanne.',
   'Charles Paul Dieudonné Armand Sabatier est doyen de la faculté des sciences.',
   'Édouard Marsal est peintre.',
   "Il se montre très favorable à la théorie de l'évolutionnisme.",
   'Il dirige la station de zoologie maritime de Sète.',
   'Il est doyen de la faculté des sciences de 1891 à 1904.',
   'Charles Paul Dieudonné Armand Sabatier est un médecin français.',
   'Charles Paul Dieudonné Armand Sabatier est mort à Montpellier.',
   "Il donne une série de cours sur l'évolutionnisme.",
   'Ce portrait est déposé à la faculté des sciences montpelliéraine.',
   'Il est enterré au cimetière protestant de Montpellier.',
   "Il est membre de l'Académie des sciences et lettres de Montpellier

In [ ]:
# Step 3: Evaluate cultural importance using OpenAI and save to JSON

def evaluate_cultural_importance(fact, language, article_name):
    """
    Use OpenAI to evaluate if a fact is culturally important for the country/culture
    associated with the given language.
    Returns only a rating: HIGH, MEDIUM, or LOW
    """
    country = LANG_COUNTRY_MAPPING.get(language, f"the {language}-speaking region")
    
    prompt = f"""Evaluate whether this fact about {article_name} is culturally important or relevant to {country}.

Fact: {fact}

Question: Is this fact particularly important or relevant to the culture, history, or context of {country}?

Respond with only one word: HIGH, MEDIUM, or LOW"""

    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": "You are an expert in cultural studies and international relations. Respond with only one word: HIGH, MEDIUM, or LOW."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=10
        )
        
        rating = response.choices[0].message.content.strip().upper()
        # Ensure it's a valid rating
        if rating not in ['HIGH', 'MEDIUM', 'LOW']:
            rating = 'MEDIUM'  # Default if response is unexpected
        return rating
    except Exception as e:
        print(f"ERROR evaluating fact: {str(e)}")
        return 'ERROR'

# Process all articles and evaluate facts
results = {}

total_articles = len(all_unmatched)
total_facts_to_evaluate = sum(
    sum(len(facts) for facts in article.values()) 
    for article in all_unmatched.values()
)

print(f"=== EVALUATING CULTURAL IMPORTANCE ===")
print(f"Total articles: {total_articles}")
print(f"Total facts to evaluate: {total_facts_to_evaluate}\n")

article_counter = 0
fact_counter = 0

for article_name, unmatched_by_lang in all_unmatched.items():
    article_counter += 1
    print(f"\n[{article_counter}/{total_articles}] Processing: {article_name}")
    print("-" * 80)
    
    results[article_name] = {}
    
    for lang, facts in unmatched_by_lang.items():
        print(f"  Language: {lang.upper()} ({len(facts)} facts)")
        results[article_name][lang] = []
        
        for fact in facts:
            fact_counter += 1
            rating = evaluate_cultural_importance(fact, lang, article_name)
            
            results[article_name][lang].append({
                'fact': fact,
                'rating': rating
            })
            
            # Print progress
            if fact_counter % 10 == 0:
                print(f"    Evaluated {fact_counter}/{total_facts_to_evaluate} facts...")
            
            # Small delay to avoid rate limiting
            time.sleep(0.3)
    
    print(f"  ✓ Completed {article_name}")

print(f"\n=== EVALUATION COMPLETE ===")
print(f"Total facts evaluated: {fact_counter}")

# Save results to JSON file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"cultural_importance_evaluation_{timestamp}.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✓ Results saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.2f} KB")

=== EVALUATING CULTURAL IMPORTANCE ===
Person: Armand Sabatier


Language: FR - Culture: France

Fact 1: Il fait ses études à Montpellier, où il suit les cours de mathématiques spéciales au lycée, puis s'inscrit en médecine.

Evaluating...
RATING: MEDIUM  
EXPLANATION: Armand Sabatier's education in Montpellier, a city known for its historical significance in medicine and academia, reflects the broader cultural value placed on education and intellectual pursuit in France. However, without additional context about his contributions or influence, this fact alone does not carry high cultural weight.

--------------------------------------------------------------------------------

Fact 2: Il donne ces cours à la faculté de théologie protestante de Montauban.

Evaluating...
RATING: MEDIUM  
EXPLANATION: Armand Sabatier's education in Montpellier, a city known for its historical significance in medicine and academia, reflects the broader cultural value placed on education and intellectual p

In [ ]:
# Step 4: Display summary statistics

import re
from collections import Counter

print("="*80)
print("=== CULTURAL IMPORTANCE SUMMARY ===")
print("="*80)

# Analyze ratings across all articles
all_ratings = []
language_ratings = {}

for article_name, langs in results.items():
    for lang, facts in langs.items():
        if lang not in language_ratings:
            language_ratings[lang] = []
        
        for item in facts:
            rating = item['rating']
            all_ratings.append(rating)
            language_ratings[lang].append(rating)

# Overall statistics
overall_counts = Counter(all_ratings)
total_facts = len(all_ratings)

print(f"\nTotal facts evaluated: {total_facts}")
print(f"\nOverall rating distribution:")
for rating in ['HIGH', 'MEDIUM', 'LOW', 'ERROR']:
    count = overall_counts.get(rating, 0)
    percentage = (count / total_facts * 100) if total_facts > 0 else 0
    print(f"  {rating}: {count} ({percentage:.1f}%)")

# Per-language statistics
print("\n" + "="*80)
print("=== RATINGS BY LANGUAGE ===")
print("="*80)

for lang in sorted(language_ratings.keys()):
    ratings = language_ratings[lang]
    counts = Counter(ratings)
    total = len(ratings)
    
    print(f"\n{lang.upper()} - {LANG_COUNTRY_MAPPING.get(lang, 'Unknown')}")
    print(f"Total facts: {total}")
    for rating in ['HIGH', 'MEDIUM', 'LOW', 'ERROR']:
        count = counts.get(rating, 0)
        percentage = (count / total * 100) if total > 0 else 0
        print(f"  {rating}: {count} ({percentage:.1f}%)")

# Top articles with HIGH importance facts
print("\n" + "="*80)
print("=== ARTICLES WITH MOST HIGH-IMPORTANCE FACTS ===")
print("="*80)

article_high_counts = {}
for article_name, langs in results.items():
    high_count = sum(
        1 for lang_facts in langs.values() 
        for item in lang_facts 
        if item['rating'] == 'HIGH'
    )
    if high_count > 0:
        article_high_counts[article_name] = high_count

# Sort by count
sorted_articles = sorted(article_high_counts.items(), key=lambda x: x[1], reverse=True)

print(f"\nTop 10 articles with HIGH importance facts:")
for i, (article, count) in enumerate(sorted_articles[:10], 1):
    print(f"  {i}. {article}: {count} HIGH-importance facts")

=== CULTURAL IMPORTANCE SUMMARY ===

FR - France
--------------------------------------------------------------------------------
Total unmatched facts: 26

Rating distribution:
  HIGH: 4 (15.4%)
  MEDIUM: 16 (61.5%)
  LOW: 6 (23.1%)
  UNKNOWN: 0 (0.0%)

HIGH importance facts (4):
  1. Ce buste est inscrit sur la liste d'objets des Monuments historiques.
  2. Ce portrait est inscrit sur la liste des objets des Monuments historiques.
  3. Durant la guerre franco-allemande de 1870, il est chirurgien responsable des ambulances du midi.
  4. Il est membre correspondant de l'Académie des sciences (1895-1910).


EN - English-speaking countries (US, UK, etc.)
--------------------------------------------------------------------------------
Total unmatched facts: 2

Rating distribution:
  HIGH: 0 (0.0%)
  MEDIUM: 0 (0.0%)
  LOW: 2 (100.0%)
  UNKNOWN: 0 (0.0%)

No HIGH importance facts found.


=== OVERALL SUMMARY ===
Total unmatched facts across all languages: 28

Overall rating distribution:
 